# Initiliaze

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

# Read Bronze table "crm_prd_info"

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
display(df)

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim((col(field.name))))

## Normalize prd_line

In [0]:

df = (
    df
    .withColumn("prd_line",
    F.when(F.lower(F.col("prd_line")) == 'm', "Mountain")
     .when(F.lower(F.col("prd_line")) == 'r', "Road")
     .when(F.lower(F.col("prd_line")) == 't', "Touring")
     .when(F.lower(F.col("prd_line")) == 's', "Other_sales")
     .otherwise("n/a")
)
)

## Product Key Parsing

In [0]:


df = df.withColumn("cat_id",F.regexp_replace(F.substring("prd_key",1,5),"-","_"))
df = df.withColumn("prd_key",F.substring("prd_key",7,F.length("prd_key")))


## clean_up prd_cost


In [0]:
df1 = spark.sql(
    """SELECT
prd_cost
FROM workspace.bronze.crm_prd_info
WHERE prd_cost IS NULL or prd_cost < 0"""
)

df = df.withColumn("prd_cost",F.coalesce(F.col("prd_cost"),F.lit(0)))


## Date Casting

In [0]:

df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

## Data Validation prd_start_dt and prd_end_dt

In [0]:
window_function = Window.partitionBy("prd_key").orderBy("prd_start_dt")
df = (
    df
      .filter(
          (F.col("prd_end_dt") < F.col("prd_start_dt")) |
          (F.col("prd_end_dt").isNull())                 
      )
      .withColumn("prd_end_dt",F.date_sub( F.lead("prd_start_dt", 1).over(window_function),1)
)
)

### Quality Check

In [0]:
df1 = df.filter((F.col("prd_end_dt") <= F.col("prd_start_dt")))
display(df1)

# Rename Collumns

In [0]:
RENAME_COLLUMNS = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}
for old_name, new_name in RENAME_COLLUMNS.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Checks of dataframe

In [0]:

df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity checks for Silver Table

In [0]:
%sql
SELECT  * 
FROM workspace.silver.crm_products LIMIT 10